# NEXUS — VS Code/Jupyter + Ollama local — REV2
Notebook local para Dias 1–5. Usa Ollama já instalado na máquina, modelos completos e os fixes de SQLite/Qwen3.

In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,time,json,re,uuid,importlib,platform,requests
def run(c,check=True,capture=False,env=None,cwd=None):return subprocess.run(c,shell=isinstance(c,str),check=check,text=True,capture_output=capture,env=env,cwd=cwd)
OVERRIDE=None;URL="https://github.com/overcyber/minicurso-mult-agents.git";DEST=Path.home()/"src/minicurso-mult-agents"
def find_repo():
    if OVERRIDE:return Path(OVERRIDE).expanduser().resolve()
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/"codigo/nexus").exists():return p
    if not DEST.exists():DEST.parent.mkdir(parents=True,exist_ok=True);run(["git","clone",URL,str(DEST)])
    return DEST
R=find_repo();N=R/"codigo/nexus";os.chdir(N);print("repo",R);print("python",sys.executable)

In [ ]:
cons=N/".nexus-jupyter-constraints.txt";cons.write_text("langchain==1.4.0\nlanggraph==1.2.11\nlanggraph-checkpoint-sqlite==3.1.1\nlangchain-ollama==1.1.0\nlangchain-chroma==1.1.0\nchromadb==1.5.9\n")
run([sys.executable,"-m","pip","install","-r",str(N/"requirements.txt"),"-r",str(N/"dia5/space_demo/requirements.txt"),"-c",str(cons),"accelerate>=1.0","ipykernel>=6.29"])
CACHE=Path(os.getenv("NEXUS_COURSE_CACHE",str(Path.home()/".cache/nexus-mult-agents-course"))).expanduser();HF=CACHE/"huggingface";RB=CACHE/"rembg";HF.mkdir(parents=True,exist_ok=True);RB.mkdir(parents=True,exist_ok=True)
os.environ.update(HF_HOME=str(HF),HF_HUB_CACHE=str(HF/"hub"),TRANSFORMERS_CACHE=str(HF/"transformers"),SENTENCE_TRANSFORMERS_HOME=str(HF/"sentence-transformers"),REMBG_HOME=str(RB))

In [ ]:
OLLAMA={"small":"qwen3:1.7b","base":"qwen3:4b","medium":"qwen3:4b","large":"qwen3:8b","embedding":"nomic-embed-text"}
HFM={"embedding":"intfloat/multilingual-e5-small","reranker":"cross-encoder/mmarco-mMiniLMv2-L12-H384-v1","sentiment":"nlptown/bert-base-multilingual-uncased-sentiment","zero_shot":"MoritzLaurer/mDeBERTa-v3-base-mnli-xnli","qa":"deepset/xlm-roberta-base-squad2","chat":"Qwen/Qwen3-0.6B"}
os.environ.update(MODELO=OLLAMA["base"],MODELO_PEQUENO=OLLAMA["small"],MODELO_MEDIO=OLLAMA["medium"],MODELO_GRANDE=OLLAMA["large"])
if not shutil.which("ollama"):raise RuntimeError("Instale Ollama no sistema e reinicie o VS Code")
BASE=os.getenv("OLLAMA_BASE_URL","http://127.0.0.1:11434").rstrip("/")
def alive():
    try:return requests.get(BASE+"/api/tags",timeout=2).ok
    except:return False
if not alive():
    proc=subprocess.Popen(["ollama","serve"],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    for _ in range(30):
        if alive():break
        time.sleep(1)
if not alive():raise RuntimeError("Ollama não responde em "+BASE)
def canon(x):
    x=(x or "").strip().lower();return x[:-7] if x.endswith(":latest") else x
def tags():return {m["name"] for m in requests.get(BASE+"/api/tags",timeout=10).json().get("models",[])}
for m in dict.fromkeys(OLLAMA.values()):
    if canon(m) not in {canon(x) for x in tags()}:run(["ollama","pull",m])
print(sorted(tags()))

In [ ]:
from huggingface_hub import snapshot_download
for repo in dict.fromkeys(HFM.values()):
    print("[HF]",repo);snapshot_download(repo_id=repo,cache_dir=os.environ["HF_HUB_CACHE"],ignore_patterns=["onnx/*","openvino/*","*.tflite","*.h5","tf_model.*","flax_model.*"])
from rembg import new_session
_s=new_session("u2net");del _s
MODS={"agente","clientes","ferramentas","indexar","nexus","hooks","steering","ferramentas_web","equipe","prompts","handoff","memoria_semantica","email_assistente","app_gradio","avaliacao","modelos","roteador","pipelines","embeddings_hf","cli"}
def dia(n):
    for m in list(sys.modules):
        if m in MODS:sys.modules.pop(m,None)
    ps=[str(N/f"dia{i}") for i in range(1,6)];sys.path[:]=[p for p in sys.path if p not in ps];sys.path.insert(0,str(N/n));importlib.invalidate_caches();os.chdir(N)
run([sys.executable,"-m","compileall","-q",str(N)]);t=run([sys.executable,"-m","pytest","-q",str(N/"testes")],False,True,cwd=str(N));print(t.stdout)
if t.returncode:raise RuntimeError(t.stderr)

# Dia 1

In [ ]:
dia("dia1");import ferramentas as f1;print(f1.calcular("950 * 24"));print(f1.ler_arquivo("../../etc/passwd"))
import agente as a1;print(a1.rodar("Qual foi o faturamento de 2024? Leia o documento necessário e cite o nome dele.",backend="ollama",verbose=True))

# Dia 2

In [ ]:
dia("dia2");import indexar as i2;b=i2.construir(recriar=not(N/".chroma").exists());print([d.metadata for d in b.similarity_search("faturamento de 2024",k=2)])
import agente as a2;print(a2.rodar("Qual foi o faturamento de 2024? Cite a fonte."))

# Dia 3

In [ ]:
dia("dia3");from langchain_core.messages import HumanMessage;import nexus as n3,sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn=sqlite3.connect(str(N/"nexus_local.db"),check_same_thread=False)
try:
    g=n3.construir_grafo().compile(checkpointer=SqliteSaver(conn),interrupt_before=[]);r=g.invoke({"messages":[HumanMessage("Qual foi o faturamento de 2024? Cite a fonte.")],"passos":0},{"configurable":{"thread_id":"local-d3"},"recursion_limit":30});print(r["messages"][-1].content)
finally:conn.close()

# Dia 4

In [ ]:
dia("dia4");from langchain_core.messages import HumanMessage;import equipe as e4,sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn=sqlite3.connect(str(N/"equipe_local.db"),check_same_thread=False)
try:
    g=e4.construir_equipe().compile(checkpointer=SqliteSaver(conn));q="Compare os fornecedores e recomende um, citando fontes.";e={"messages":[HumanMessage(q)],"pergunta":q,"proximo":"","instrucao":q,"achados":[],"rascunho":"","veredito":"","rodadas":0};r=g.invoke(e,{"configurable":{"thread_id":"local-d4"},"recursion_limit":40});print(r.get("veredito"));print(r.get("rascunho"))
finally:conn.close()

# Dia 5

In [ ]:
import torch;dev="cuda" if torch.cuda.is_available() else "cpu";pdev=0 if torch.cuda.is_available() else -1
dia("dia5");import modelos as m5;exp={"supervisor":OLLAMA["small"],"pesquisador":OLLAMA["medium"],"analista":OLLAMA["small"],"redator":OLLAMA["large"],"critico":OLLAMA["medium"]}
for p,x in exp.items():print(p,m5.para(p).model);assert m5.para(p).model==x
from langchain_huggingface import HuggingFaceEmbeddings
emb=HuggingFaceEmbeddings(model_name=HFM["embedding"],model_kwargs={"device":dev},encode_kwargs={"normalize_embeddings":True});print(len(emb.embed_query("faturamento 2024")))

In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer
tok=AutoTokenizer.from_pretrained(HFM["chat"]);model=AutoModelForCausalLM.from_pretrained(HFM["chat"],torch_dtype="auto",device_map="auto" if torch.cuda.is_available() else None)
msgs=[{"role":"system","content":"Responda em português e objetivamente."},{"role":"user","content":"Explique em duas frases o papel de um supervisor multiagente."}]
inp=tok.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,enable_thinking=False,return_tensors="pt",return_dict=True);inp={k:v.to(model.device) for k,v in inp.items()}
with torch.inference_mode():out=model.generate(**inp,max_new_tokens=180,do_sample=False,pad_token_id=tok.eos_token_id)
txt=tok.decode(out[0][inp["input_ids"].shape[-1]:],skip_special_tokens=True).strip();assert "<think>" not in txt;print(txt)

In [ ]:
# Avaliação opcional com SQLite válido.
RODAR_10=False
dia("dia4");import equipe as ev,sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import HumanMessage
conn=None
def responder(q):
    global conn
    if conn is None:conn=sqlite3.connect(str(N/"equipe_avaliacao_local.db"),check_same_thread=False)
    g=ev.construir_equipe().compile(checkpointer=SqliteSaver(conn));e={"messages":[HumanMessage(q)],"pergunta":q,"proximo":"","instrucao":q,"achados":[],"rascunho":"","veredito":"","rodadas":0};r=g.invoke(e,{"configurable":{"thread_id":f"eval-{uuid.uuid4()}"},"recursion_limit":40});txt=r.get("rascunho") or r["messages"][-1].content or "";return txt,sorted(set(re.findall(r"\[fonte:\s*([^\]]+)\]",txt,re.I))),0
dia("dia5");import avaliacao as av;casos=av.carregar_casos(N/"avaliacao/casos.jsonl");print("casos",len(casos))
try:
    if RODAR_10:linhas=av.rodar(responder,casos);print(av.tabela(linhas))
finally:
    if conn is not None:conn.close()